# Analyze real datasets with the server pipeline

This notebook runs `Datas_Scripts.Run_Datas_Analysis.run_split` on Crime,
COMPAS, Student, Law, and Adult. The saved results are read back for
hyperparameter summaries and four rho/lambda metric tables.

The notebook and server command use the same data preparation, GP fitting,
fairness optimization, and pseudoinverse prediction functions. The quick
configuration below uses up to 100 training and 200 test observations per dataset,
one split seed, two hyperparameter starts, and short iteration budgets.
Use longer runs and repeated split seeds for research comparisons.

## Download the raw datasets

From the repository root, run the following command once in a terminal,
then run this notebook's cells in order.

```bash
python Datas_Scripts/download_datasets.py
```

The helper retrieves the five datasets into `Datas_Scripts/data/`.
Use `--datasets law student` to retrieve selected datasets. If the files
are stored elsewhere, pass `--data-dir /path/to/data` to the download helper
and set `DATA_DIR` in the next cell to that directory.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

root = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "Fair_GPs" / "Fair_GPs_classe.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the repository or its notebook directory.")
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch as t
from IPython.display import display

from Datas_Scripts.Run_Datas_Analysis import run_split
from Fair_GPs.Data_Processing import SolutionAnalyzer

t.set_default_dtype(t.float64)
t.set_default_device("cpu")
t.set_num_threads(2)

DATA_DIR = root / "Datas_Scripts" / "data"  # Edit when the raw files are stored elsewhere.
DATASETS = ("crime", "compas", "student", "law", "adult")
SPLIT_SEED = 0
REPORT_DATASET = "student"

CONFIG = {
    "device": "cpu",
    "threads": 2,
    "train_cap": 100,
    "test_cap": 200,
    "gp_seed": 42,
    "gp_iters": 20,
    "gp_lr": 0.005,
    "doe_points": 1,
    "design": "sobol",
    "fixed_rhos": [-1.0, -0.9, 0.0, 0.9, 1.0],
    "rho_initial": 0.0,
    "methods": ["MMD", "Sinkhorn"],
    "lambda_count": 3,
    "lambda_min_exp": -1.0,
    "lambda_max_exp": 9.0,
    "mmd_iters": [20, 40],
    "sinkhorn_iters": [15, 30],
    "opt_lr": [0.005, 0.0005],
    "sinkhorn_blur": 0.05,
    "prediction_jitter": 0.0,
    "prediction_rtol": None,
    "prediction_atol": 0.0,
    "prediction_batch_size": 512,
    "inducing_m": 400,
    "sparse_threshold": 7000,
    "save_artifacts": True,
    "save_kernel_inverse": False,
}

## 1. Data and reproducible split membership

All datasets use a random 70/30 split with the selected seed, followed by
sensitive-group-stratified subsampling inside each pool when a cap applies.
Adult's source files are pooled before this repeated-split procedure. Law
retains all available rows and labels White individuals as S=1 and the
other known race categories as S=0.

Continuous features use training-fitted scaling to `[0, 1]`. Category codes
and the final binary sensitive label are preserved. Targets are centered
at the selected training mean and divided by the largest absolute training
deviation, then clipped to `[-1, 1]` on both splits. COMPAS and Adult are
evaluated as regression problems with binary targets.

The temporary result directory remains available while this notebook's
kernel is running. Restarting the kernel removes it. The cells read the
saved data and results using the same artifact layout as the server.

`Datas_Scripts.Data_Preprocessing.load_dataset(name, DATA_DIR)` reads the
raw files and assembles continuous and categorical feature columns with
the binary sensitive label last. `run_split` calls that loader and applies
the shared split, normalization, fitting, optimization, and prediction
functions. The next cell gives each call a fresh output directory.


In [ ]:
temporary_parent = root / "results"
temporary_parent.mkdir(exist_ok=True)
temporary_results = TemporaryDirectory(prefix="notebook_analysis_", dir=temporary_parent)
OUTPUT_DIR = Path(temporary_results.name)

job_summaries, datasets, metric_frames, model_frames = [], {}, [], []
for name in DATASETS:
    folder = OUTPUT_DIR / name / f"split_{SPLIT_SEED:03d}"
    summary = run_split(name, SPLIT_SEED, CONFIG, DATA_DIR, folder)
    job_summaries.append(summary)
    datasets[name] = t.load(folder / "split.pt", map_location="cpu", weights_only=True)
    metric_frames.append(pd.read_csv(folder / "metrics.csv"))
    model_frames.append(pd.read_csv(folder / "models.csv"))

all_metrics = pd.concat(metric_frames, ignore_index=True)
hyperparameters = pd.concat(model_frames, ignore_index=True)
split_audit = pd.DataFrame([
    {"dataset": name, "split": split,
     "pool_n": len(data[f"{split}_pool_indices"]),
     "used_n": len(data[f"{split}_indices"]),
     "S=0": int((data[f"XS_{split}"][:, -1] == 0).sum()),
     "S=1": int((data[f"XS_{split}"][:, -1] == 1).sum()),
     "categorical features": ", ".join(data["feature_names"][i] for i in data["categorical_dims"])}
    for name, data in datasets.items() for split in ("train", "test")
])
display(split_audit)
display(pd.DataFrame(job_summaries))

## 2. Read the saved fairness paths

The server fits the five fixed correlations and one training-estimated rho.
Each fit uses a baseline start plus the configured Sobol design points.
The categorical correlation is estimated for datasets containing nominal
features. The training prediction error and Wasserstein barycenter cost
calibrate the noise variance used in the quadratic fairness objective.

Both penalties use the same relative weights, zero followed by
`exp(linspace(-1, 9, lambda_count))`. Absolute weights multiply these
factors by the Frobenius norm of the fitted posterior precision. Each
positive weight starts from the preceding solution. MMD uses the
training-posterior median bandwidth, and Sinkhorn uses the configured blur.

For each method and model, the saved path contains optimized latent values,
observed and paired counterfactual predictions, objectives, and iteration
logs. The GP prediction uses the cached spectral pseudoinverse with the
configured cutoff.

In [ ]:
results = {}
for name in DATASETS:
    folder = OUTPUT_DIR / name / f"split_{SPLIT_SEED:03d}"
    model_rows = hyperparameters.loc[hyperparameters.dataset.eq(name)].sort_values(
        ["rho", "rho_estimated"]
    )
    paths = {
        key: {
            method: t.load(folder / "models" / key / "paths" / f"{method}.pt",
                           map_location="cpu", weights_only=True)
            for method in CONFIG["methods"]
        }
        for key in model_rows.model_key
    }
    results[name] = {"models": model_rows, "paths": paths}

expected_rows = len(DATASETS) * len(CONFIG["methods"]) * 2 * (
    len(CONFIG["fixed_rhos"]) + 1
) * (CONFIG["lambda_count"] + 1)
assert len(all_metrics) == expected_rows
assert np.isfinite(all_metrics[["MSE", "KS", "disp", "cf_max_abs"]].to_numpy()).all()
assert all_metrics.loc[all_metrics.model_key.eq("rho_1"), "disp"].max() < 1e-16

display(hyperparameters.loc[hyperparameters.dataset.eq(REPORT_DATASET)])

## 3. Compare train and test predictions

Each table displays MSE on the centered target scale, observed-group KS,
and the mean squared difference of matched counterfactual predictions
(`disp`). The exported `cf_max_abs` metric records the maximum absolute
counterfactual difference.

`SolutionAnalyzer` reads the saved path predictions to reconstruct every
table. Its values are checked against the server's emitted metric CSV.
Set `REPORT_DATASET` to view another dataset's four tables. The
`all_metrics`, `hyperparameters`, and `split_audit` dataframes contain
all completed datasets.

In [ ]:
analyzers = {}
for name, result in results.items():
    model_rows = result["models"]
    keys = model_rows.model_key.tolist()
    rhos = model_rows.rho.tolist()
    for method in CONFIG["methods"]:
        paths = [result["paths"][key][method] for key in keys]
        lambdas = t.stack([path["lambdas"] for path in paths])
        relative = paths[0]["lambda_relative"]
        for split in ("train", "test"):
            observed = t.stack([path["predictions"][split]["observed"] for path in paths])
            cf0 = t.stack([path["predictions"][split]["cf0"] for path in paths])
            cf1 = t.stack([path["predictions"][split]["cf1"] for path in paths])
            data = datasets[name]
            analyzer = SolutionAnalyzer(
                observed, lambdas, data[f"XS_{split}"][:, -1], data[f"Y_{split}"],
                rhos=rhos, counterfactuals=(cf0, cf1),
            )
            analyzers[name, method, split] = analyzer
            rebuilt = analyzer.compute_metrics()
            rebuilt["model_key"] = [keys[index] for index in rebuilt.rho_index]
            saved = all_metrics.loc[
                all_metrics.dataset.eq(name) & all_metrics.method.eq(method) & all_metrics.split.eq(split)
            ]
            comparison = rebuilt.merge(saved, on=["model_key", "lambda_index"], suffixes=("_table", "_saved"))
            for metric in ("MSE", "KS", "disp"):
                np.testing.assert_allclose(comparison[f"{metric}_table"], comparison[f"{metric}_saved"],
                                           rtol=1e-12, atol=1e-12)
            if name == REPORT_DATASET:
                fig, ax = analyzer.plot_metrics_table(
                    title=f"{name.capitalize()} · {method} · {split}",
                    lambda_factors=relative.numpy(), digits=4,
                )
                for index, row in enumerate(model_rows.itertuples()):
                    if row.rho_estimated:
                        label = next(text for text in ax.texts if text.get_position() == (0.5, index + 1.5))
                        label.set_text(f"ρ*={row.rho:.6f}")
                plt.show()

print(f"Verified {len(all_metrics)} metric rows against the saved server predictions.")